# 🏆 LifeMate - Zalo AI Challenge 2026 (Kaggle 2x Tesla T4 GPUs)

Notebook này được thiết kế riêng để chạy pipeline Agentic Voice Assistant trên **Kaggle Notebooks với 2x GPU Tesla T4 (Tổng cộng ~30GB VRAM)**.

---

### ⚡ Tối ưu song song 2 GPU (Data Parallelism):
- **Cả 2 GPU hoạt động 100% công suất đồng thời**: GPU 0 xử lý nửa đầu (Shard 0), GPU 1 xử lý nửa sau (Shard 1).
- **Tốc độ nhanh gấp đôi**: Toàn bộ 634 mẫu `public_test` chỉ mất **khoảng 15 - 20 phút**!
- **Không lo OOM**: Mỗi GPU có 15GB VRAM, mô hình 4-bit chỉ chiếm ~2.8GB (dư tới 12GB VRAM mỗi GPU).

---

### ⚙️ Thiết lập quan trọng trên Kaggle trước khi chạy:
Ở thanh menu bên phải (**Session options**):
1. **ACCELERATOR**: Chọn **`GPU T4 x2`**
2. **INTERNET**: Bật **`Internet On`** (Bắt buộc bật để clone Git và tải weights mô hình)
3. **INPUT**: Nhấn **`+ Add Data`** để thêm dataset `public_test` mà bạn đã upload lên Kaggle Datasets.

## Bước 1: Kiểm tra 2x GPU Tesla T4

In [ ]:
# Kiểm tra cả 2 GPU Tesla T4 (GPU 0 và GPU 1)
import torch
print(f"🔥 Phát hiện {torch.cuda.device_count()} GPU:")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"   • GPU {i}: {p.name} ({p.total_memory / (1024**3):.1f} GB VRAM)")

!nvidia-smi

## Bước 2: Clone Codebase từ Git Repo

In [ ]:
import os

# Di chuyển vào thư mục làm việc ghi được (/kaggle/working)
%cd /kaggle/working
!rm -rf LifeMate

# Thay <URL_REPO_CỦA_BẠN> bằng URL git của bạn
REPO_URL = "https://github.com/your-username/LifeMate.git"  # <-- Cập nhật URL repo của bạn
BRANCH = "colab-t4"                                         # <-- Branch mới nhất

!git clone -b $BRANCH $REPO_URL LifeMate
%cd /kaggle/working/LifeMate

## Bước 3: Cài đặt thư viện phụ thuộc

In [ ]:
# Cài đặt các package cần thiết: transformers, accelerate, bitsandbytes, qwen-agent, sherpa-onnx
!pip install -q -r requirements.txt

## Bước 4: Tự động tìm đường dẫn Dataset trong `/kaggle/input`

In [ ]:
from pathlib import Path

# Quét thư mục chứa public_test trong /kaggle/input
kaggle_input = Path("/kaggle/input")
found_dirs = list(kaggle_input.rglob("public_test.jsonl"))

if found_dirs:
    data_dir = str(found_dirs[0].parent)
    print(f"✅ Đã tìm thấy thư mục dữ liệu public_test: {data_dir}")
else:
    # Fallback thủ công nếu bạn đặt tên dataset khác
    data_dir = "/kaggle/input/zaic2026-public-test/public_test"  # <-- Sửa đường dẫn nếu cần
    print(f"⚠️ Không tự động tìm thấy, sử dụng đường dẫn mặc định: {data_dir}")

os.environ["DATA_DIR"] = data_dir

## Bước 5: Chạy thử nghiệm 2 mẫu đầu tiên (Sanity Check)
Chạy thử 2 mẫu để xác nhận môi trường và inference trơn tru.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!python main.py \
    --mode public_test \
    --num-samples 2 \
    --data-dir "$DATA_DIR" \
    --output /kaggle/working/test_submission.jsonl

## Bước 6: Khởi chạy song song 2 worker trên GPU 0 và GPU 1 (Data Parallelism)
Chạy ngầm đồng thời 2 tiến trình độc lập:
- **GPU 0**: Xử lý Shard 0 (317 mẫu đầu) -> lưu vào `/kaggle/working/sub_part0.jsonl`
- **GPU 1**: Xử lý Shard 1 (317 mẫu sau) -> lưu vào `/kaggle/working/sub_part1.jsonl`
- Cell này khởi chạy ngầm trong **vài giây** và không khóa notebook, sau đó bạn có thể chạy Bước 7 để theo dõi realtime!

In [ ]:
%%bash
# Đảm bảo đứng đúng thư mục repo
cd /kaggle/working/LifeMate
export PYTHONUNBUFFERED=1
export PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"

# Xóa hoặc dọn dẹp file log cũ
> /kaggle/working/gpu0.log
> /kaggle/working/gpu1.log

# Khởi chạy GPU 0 (Shard 0: 317 mẫu đầu)
CUDA_VISIBLE_DEVICES=0 python -u main.py \
    --mode public_test \
    --num-samples -1 \
    --data-dir "$DATA_DIR" \
    --num-shards 2 \
    --shard-id 0 \
    --output /kaggle/working/sub_part0.jsonl >> /kaggle/working/gpu0.log 2>&1 &

# Khởi chạy GPU 1 (Shard 1: 317 mẫu sau)
CUDA_VISIBLE_DEVICES=1 python -u main.py \
    --mode public_test \
    --num-samples -1 \
    --data-dir "$DATA_DIR" \
    --num-shards 2 \
    --shard-id 1 \
    --output /kaggle/working/sub_part1.jsonl >> /kaggle/working/gpu1.log 2>&1 &

sleep 3
echo "🚀 Đã khởi chạy thành công 2 GPU workers ngầm!"
ps aux | grep "python -u main.py" | grep -v grep || true


## Bước 7: Theo dõi tiến độ thời gian thực (Chạy cell này bất kỳ lúc nào)
Cell này hiển thị số tiến trình đang chạy, số dòng kết quả đã hoàn thành và log mới nhất của mỗi GPU.

In [ ]:
print("=== TRẠNG THÁI TIẾN TRÌNH ===")
!ps aux | grep "python.*main.py" | grep -v grep || echo "Không có tiến trình main.py nào đang chạy."

print("\n=== SỐ DÒNG ĐÃ HOÀN THÀNH (Mỗi shard ~317 mẫu) ===")
!wc -l /kaggle/working/sub_part0.jsonl /kaggle/working/sub_part1.jsonl 2>/dev/null || true

print("\n=== LOG GẦN NHẤT GPU 0 ===")
!tail -n 8 /kaggle/working/gpu0.log 2>/dev/null || echo "(Chưa có log)"

print("\n=== LOG GẦN NHẤT GPU 1 ===")
!tail -n 8 /kaggle/working/gpu1.log 2>/dev/null || echo "(Chưa có log)"


## Bước 8: Đợi hoàn tất & Gộp kết quả thành submission.jsonl
Chạy cell này để đợi cả 2 GPU xử lý xong hoàn toàn, sau đó tự động gộp 2 file shard thành `submission.jsonl`.

In [ ]:
%%bash
echo "⏳ Đang chờ 2 worker GPU hoàn tất..."
while pgrep -f "python.*main.py" > /dev/null; do
    sleep 10
done

echo "🎉 Cả 2 GPU đã chạy xong! Đang gộp file submission.jsonl..."
cat /kaggle/working/sub_part0.jsonl /kaggle/working/sub_part1.jsonl > /kaggle/working/submission.jsonl

echo "✅ Tổng số dòng trong submission.jsonl:"
wc -l /kaggle/working/submission.jsonl

head -n 2 /kaggle/working/submission.jsonl
echo "🚀 File submission.jsonl đã sẵn sàng để nộp bài!"
